# 📈 Finance-Pro: Quantitative ML Model Training & Evaluation Pipeline

Notebook ini menyediakan pipeline end-to-end terstruktur untuk melatih dan mengevaluasi seluruh model Machine Learning pada sistem **Finance-Pro v7**:
1. **Load Data**: Mengambil data OHLCV historis dari SQLite database (`data/ihsg_trading.db`).
2. **Data Cleaning**: Imputasi missing values, handling split adjustment, dan filter gocap.
3. **Feature Engineering**: Ekstraksi 29+ fitur teknikal (Malla et al.) & Triple Barrier Meta-Labeling.
4. **Model Training & Walk-Forward Validation**: Pelatihan model XGBoost, LightGBM, Ensemble, dan Supervised Autoencoder.
5. **Evaluation**: Evaluasi performa Out-of-Sample (Accuracy, F1-macro, Log-loss, AUC-OVR) & Feature Importances.

## Cell 1: Environment Setup & Imports

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Ensure project root is in path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from pipeline.storage import StorageManager
from pipeline.data_cleaner import DataCleaner
from shared.features.feature_builder import FeatureBuilder
from model.xgboost_trainer import XGBoostTrainer, XGBoostConfig
from model.lightgbm_trainer import LightGBMTrainer, LightGBMConfig
from model.autoencoder import SupervisedAutoencoder, AutoencoderConfig
from model.ensemble import EnsembleModel, EnsembleConfig
from model.walk_forward import WalkForwardValidator
from model.evaluator import ModelEvaluator
from model.trainer import ModelTrainer, TrainingConfig
from model.registry import ModelRegistry

print("✓ Environment & modules successfully loaded!")

## Cell 2: Load Data dari SQLite Database

In [ ]:
storage = StorageManager()
available_tickers = storage.get_available_tickers()
date_range = storage.get_date_range()

print(f"Database Path  : {storage.db_path}")
print(f"Total Tickers  : {len(available_tickers)} emitens")
print(f"Date Coverage  : {date_range[0]} to {date_range[1]}")

# Load raw close prices
raw_close_prices = storage.load_close_prices()
print(f"Raw Close Data Shape: {raw_close_prices.shape}")
raw_close_prices.head()

## Cell 3: Data Cleaning & Preprocessing

In [ ]:
cleaner = DataCleaner(min_price=200.0) # Filter saham gocap (< Rp200)
cleaned_prices = cleaner.clean(raw_close_prices)

print(f"Cleaned Data Shape: {cleaned_prices.shape}")
print(f"Remaining NaNs    : {cleaned_prices.isna().sum().sum()}")
cleaned_prices.tail()

## Cell 4: Feature Engineering & Labeling

In [ ]:
fb = FeatureBuilder()

# Build multi-feature matrix per ticker
feature_matrices = []
labels_list = []
ret = cleaned_prices.pct_change().shift(-1)

for ticker in cleaned_prices.columns:
    ohlc = pd.DataFrame({
        "open": cleaned_prices[ticker],
        "high": cleaned_prices[ticker] * 1.002,
        "low": cleaned_prices[ticker] * 0.998,
        "close": cleaned_prices[ticker],
        "volume": 1000000
    })
    tf = fb.build_technical_features(ohlc).dropna()
    lbl = np.where(ret[ticker].loc[tf.index] > 0.005, 1, np.where(ret[ticker].loc[tf.index] < -0.005, -1, 0))
    
    feature_matrices.append(tf.values)
    labels_list.append(lbl)

X = np.vstack(feature_matrices)
y = np.concatenate(labels_list)

# Map labels (-1, 0, 1) -> (0, 1, 2) [0=LOSS, 1=NEUTRAL, 2=PROFIT]
if -1 in y:
    y = y + 1

sample_ohlc = pd.DataFrame({
    "open": cleaned_prices.iloc[:, 0], "high": cleaned_prices.iloc[:, 0], "low": cleaned_prices.iloc[:, 0], "close": cleaned_prices.iloc[:, 0], "volume": 1000000
})
feature_names = list(fb.build_technical_features(sample_ohlc).dropna().columns)

print(f"X Feature Matrix Shape : {X.shape} ({len(feature_names)} features)")
print(f"y Label Vector Shape   : {y.shape}")
print(f"Class Distribution     : Loss(0)={np.sum(y==0)}, Neutral(1)={np.sum(y==1)}, Profit(2)={np.sum(y==2)}")

## Cell 5: Model Training & Walk-Forward Validation

Di cell ini kita melatih 4 arsitektur model secara berurutan dengan **Walk-Forward Validation** (train_size=5000, test_size=1000):

In [ ]:
# Take last 20,000 samples for efficient notebook evaluation
X_eval = X[-20000:]
y_eval = y[-20000:]

wf = WalkForwardValidator(mode="expanding", train_size=5000, test_size=1000, max_folds=3)
models = {
    "XGBoost": XGBoostTrainer(XGBoostConfig(n_estimators=100, learning_rate=0.05)),
    "LightGBM": LightGBMTrainer(LightGBMConfig(n_estimators=100, learning_rate=0.05)),
    "Supervised Autoencoder": SupervisedAutoencoder(AutoencoderConfig(encoding_dim=12)),
    "Ensemble (Weighted)": EnsembleModel(EnsembleConfig(method="weighted", use_xgb=True, use_lgbm=True)),
}

trained_models = {}
val_results = {}

for name, model in models.items():
    print(f"\n🔄 Training & Validating Model: {name}...")
    res = wf.validate(model, X_eval, y_eval, fit_kwargs={"feature_names": feature_names} if "XGBoost" in name else {})
    val_results[name] = res
    trained_models[name] = model
    print(f"✓ {name} Completed — Mean Accuracy: {res.aggregate_metrics.get('accuracy', 0):.4f}, F1-Macro: {res.aggregate_metrics.get('f1_macro', 0):.4f}")

## Cell 6: Detailed Evaluation & Model Comparison

In [ ]:
evaluator = ModelEvaluator()

summary_rows = []
for name, res in val_results.items():
    m = res.aggregate_metrics
    summary_rows.append({
        "Model": name,
        "Accuracy": m.get("accuracy", 0),
        "F1 Macro": m.get("f1_macro", 0),
        "F1 Weighted": m.get("f1_weighted", 0),
        "Log Loss": m.get("log_loss", 0),
        "AUC OVR": m.get("auc_ovr", 0),
    })

summary_df = pd.DataFrame(summary_rows).set_index("Model")
print("\n=================== MODEL COMPARISON TABLE ===================")
display(summary_df)

# Feature Importance Visualization (XGBoost)
xgb_model = trained_models["XGBoost"]
if hasattr(xgb_model, "feature_importances_") and xgb_model.feature_importances_ is not None:
    importances = xgb_model.feature_importances_
    top_indices = np.argsort(importances)[-10:]
    plt.figure(figsize=(10, 5))
    plt.barh([feature_names[i] for i in top_indices], importances[top_indices], color='#88C0D0')
    plt.title("Top 10 Feature Importances (XGBoost)", fontsize=14, color='#2E3440')
    plt.xlabel("Importance Score")
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

## Cell 7: Save & Register Model Artifacts to Model Registry

Model terbaik akan otomatis disimpan ke `artifacts/saved_models/` **dan** terdaftar di `ModelRegistry` (`artifacts/registry.db`).  
Model dengan F1-Macro tertinggi akan secara otomatis di-**promote** ke stage **PRODUCTION** agar langsung dipakai oleh TUI Scanner.

In [ ]:
output_dir = os.path.join(PROJECT_ROOT, "artifacts", "saved_models")
os.makedirs(output_dir, exist_ok=True)

registry = ModelRegistry(
    artifacts_dir=output_dir,
    db_path=os.path.join(PROJECT_ROOT, "artifacts", "registry.db"),
)

best_f1 = -1
best_version_id = None

for name, model in trained_models.items():
    filename = f"{name.lower().replace(' ', '_').replace('(', '').replace(')', '')}_notebook.pkl"
    save_path = os.path.join(output_dir, filename)
    if hasattr(model, "save"):
        model.save(save_path)
        print(f"✓ Saved artifact: {save_path}")

        # Auto-register ke ModelRegistry
        model_type = name.lower().split()[0]  # 'xgboost', 'lightgbm', etc.
        metrics = val_results[name].aggregate_metrics
        mv = registry.register(
            model_type=model_type,
            artifact_path=save_path,
            metrics=metrics,
            description=f"Trained from Notebook 01 ({name})",
        )
        print(f"  📦 Registered as: {mv.version_id} (stage={mv.stage})")

        # Track best model by F1-Macro
        f1 = metrics.get("f1_macro", 0)
        if f1 > best_f1:
            best_f1 = f1
            best_version_id = mv.version_id

# Auto-promote best model to PRODUCTION
if best_version_id:
    registry.promote(best_version_id)
    print(f"\n🟢 Best model '{best_version_id}' promoted to PRODUCTION (F1-Macro: {best_f1:.4f})")

# Show final registry state
print("\n📦 Current Model Registry:")
for mv in registry.list_versions():
    print(f"  {mv.version_id:20s} | {mv.stage:12s} | Acc={mv.metrics.get('accuracy',0):.4f} | F1={mv.metrics.get('f1_macro',0):.4f}")